# Phase 2 — EDA / Visualisation

This notebook uses the cleaned data and documents plot-level takeaways.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

clean_path = "dcs_student_data_cleaned.csv"
raw_path = "dcs_student_data.csv"

if pd.io.common.file_exists(clean_path):
    df = pd.read_csv(clean_path)
    source = clean_path
else:
    df = pd.read_csv(raw_path)

    score_columns = [
        "Attendance (%)",
        "Midterm_Score",
        "Final_Score",
        "Assignments_Avg",
        "Quizzes_Avg",
        "Participation_Score",
        "Projects_Score",
        "Total_Score",
        "math_score",
        "reading_score",
        "writing_score",
        "science_score",
    ]

    for col in score_columns + ["Age"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    dept_map = {
        "engineering": "Engineering",
        "business": "Business",
        "mathematics": "Mathematics",
        "math": "Mathematics",
        "computer science": "Computer Science",
        "cs": "Computer Science",
    }
    df["Gender"] = df["Gender"].astype(str).str.strip().str.lower().map({
        "female": "Female",
        "male": "Male",
    }).fillna(df["Gender"].astype(str).str.strip().str.title())
    df["Department"] = (
        df["Department"].astype(str).str.strip().str.lower().map(dept_map)
        .fillna(df["Department"].astype(str).str.strip().str.title())
    )

    for col in score_columns:
        df.loc[(df[col] < 0) | (df[col] > 100), col] = np.nan
    df.loc[(df["Age"] < 15) | (df["Age"] > 60), "Age"] = np.nan
    df = df.drop_duplicates(subset=["Student_ID"], keep="first").copy()

    source = raw_path + " (cleaned in notebook)"

print(f"Loaded data source: {source}")
print(f"Shape used for EDA: {df.shape}")


In [ ]:
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
plt.figure(figsize=(14, 10))
sns.heatmap(df[numeric_cols].corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Numeric Columns)")
plt.tight_layout()
plt.show()


**Takeaway:** Most correlations are near zero, indicating little to no predictive linear signal between features and overall scores in this dataset.


In [ ]:
score_columns = [
    "Midterm_Score",
    "Assignments_Avg",
    "Quizzes_Avg",
    "Participation_Score",
    "Projects_Score",
    "math_score",
    "reading_score",
    "writing_score",
    "science_score",
]

for target in ["Final_Score", "Total_Score"]:
    fig, axes = plt.subplots(3, 3, figsize=(16, 14))
    axes = axes.flatten()

    for ax, col in zip(axes, score_columns):
        sns.scatterplot(data=df, x=col, y=target, alpha=0.4, s=20, ax=ax)
        corr = df[[col, target]].corr().iloc[0, 1]
        ax.set_title(f"{col} vs {target} (r={corr:.3f})")

    plt.tight_layout()
    plt.show()


**Takeaway:** Scatter plots remain diffuse across all feature-score pairs, and the correlation values are close to zero for most relationships.


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="Department", y="Total_Score")
plt.xticks(rotation=20)
plt.title("Total_Score by Department")
plt.tight_layout()
plt.show()


**Takeaway:** Department-level score distributions overlap heavily, with no strong separation in medians or spread.


In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x="Gender", y="Total_Score")
plt.title("Total_Score by Gender")
plt.tight_layout()
plt.show()


**Takeaway:** Gender groups show very similar score distributions, suggesting no meaningful difference by gender in this dataset.


In [ ]:
attendance_final_corr = df[["Attendance (%)", "Final_Score"]].corr().iloc[0, 1]
attendance_total_corr = df[["Attendance (%)", "Total_Score"]].corr().iloc[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x="Attendance (%)", y="Final_Score", alpha=0.4, s=20, ax=axes[0])
axes[0].set_title(f"Attendance vs Final_Score (r={attendance_final_corr:.3f})")

sns.scatterplot(data=df, x="Attendance (%)", y="Total_Score", alpha=0.4, s=20, ax=axes[1])
axes[1].set_title(f"Attendance vs Total_Score (r={attendance_total_corr:.3f})")

plt.tight_layout()
plt.show()

print(f"Attendance vs Final_Score correlation: {attendance_final_corr:.3f}")
print(f"Attendance vs Total_Score correlation: {attendance_total_corr:.3f}")


**Takeaway:** Attendance has near-zero correlation with both Final_Score and Total_Score in this data, so it does not appear to explain performance variation.


## Final EDA Conclusion

The heatmap and all scatter/box plots consistently show a **no-signal pattern**: variables are largely independent with weak correlations and overlapping distributions, so strong predictive performance should not be expected from these inputs.
